# 🎨 Image Restoration Dual-Model Training - Colab

This notebook supports **Deterministic (SwinIR)** and **Generative (SD-LoRA)** restoration with automatic **Benchmarking**.

### ✅ Requirements:
Ensure you have uploaded **BOTH** `restoration_code.zip` and `processed_data.zip` to the root of your Google Drive.

### 1. Mount Google Drive & Environment Setup

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

# Create directories for weights and validation logs
!mkdir -p /content/drive/MyDrive/weights/swinir_checkpoints
!mkdir -p /content/drive/MyDrive/weights/sd_lora_val_images

### 2. Smart Extract: Code & Data

In [ ]:
# 1. Extract Source Code (Fastest)
print("📁 Extracting Code...")
!unzip -q /content/drive/MyDrive/restoration_code.zip -d /content/

# 2. Extract Large Dataset
if os.path.exists('/content/drive/MyDrive/processed_data.zip'):
    print("📊 Extracting Processed Dataset...")
    !unzip -q /content/drive/MyDrive/processed_data.zip -d /content/
else:
    print("⚠️ 'processed_data.zip' not found. You will need to run Data Preparation in Step 3.")

%cd /content/training

# Install dependencies (fast cloud install)
!pip install -q -r requirements.txt
!pip install -q xformers peft accelerate diffusers transformers

### 3. Data Preparation (ONLY if processed_data.zip was missing)
This generates clean/damaged pairs and separates them into `train` and `test` sets automatically.

In [ ]:
# !python scripts/prepare_data.py --input ./datasets/raw_lite \
#                               --output ./datasets/processed \
#                               --seed 42 \
#                               --split_ratio 0.9 \
#                               --multiplier 1

### 4. Track 1: SwinIR-Light Training (Sanity Test)
Run a quick 1-epoch test to ensure the dataset is loaded correctly and CUDA is working before the full run.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --epochs 1 \
                                --batch_size 2 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_test.pth

### 4.5 Track 1: SwinIR-Light Training (Full Run)
The model will periodically evaluate on the test set and save the best PSNR weights.

In [ ]:
!python scripts/train_swinir.py --clean_dir ./datasets/processed/train/clean \
                                --damaged_dir ./datasets/processed/train/damaged \
                                --val_clean_dir ./datasets/processed/test/clean \
                                --val_damaged_dir ./datasets/processed/test/damaged \
                                --epochs 100 \
                                --batch_size 8 \
                                --patch_size 128 \
                                --weight_path /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                --resume

### 5. Track 2: SD-LoRA Transfer Training (Sanity Test)
Run a quick 1-epoch test with a tiny SD model to ensure the pipeline works.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_test \
                                 --epochs 1 \
                                 --batch_size 1 \
                                 --model_id hf-internal-testing/tiny-stable-diffusion-torch

### 5.5 Track 2: SD-LoRA Transfer Training (Full Run)
Check `/content/drive/MyDrive/weights/sd_lora_val_images` to see restoration progress.

In [ ]:
!python scripts/train_sd_lora.py --image_dir ./datasets/processed/train/clean \
                                 --output_dir /content/drive/MyDrive/weights/sd_lora_antique_colab \
                                 --val_dir /content/drive/MyDrive/weights/sd_lora_val_images \
                                 --epochs 20 \
                                 --batch_size 4 \
                                 --resume

### 6. Track 3: Benchmarking & Comparison
Run metrics and generate a side-by-side [ GT | Damaged | SwinIR | AI ] comparison.

In [ ]:
print("📊 Running Quantitative Metrics (PSNR/SSIM)...")
!python scripts/evaluate.py --clean_dir ./datasets/processed/test/clean \
                            --damaged_dir ./datasets/processed/test/damaged \
                            --weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth

print("🖼️ Generating visual comparison grid...")
!python scripts/test_restoration.py --test_clean_dir ./datasets/processed/test/clean \
                                    --test_damaged_dir ./datasets/processed/test/damaged \
                                    --swinir_weights /content/drive/MyDrive/weights/swinir_restoration_colab.pth \
                                    --lora_path /content/drive/MyDrive/weights/sd_lora_antique_colab/latest \
                                    --output_dir ./test_results \
                                    --num_test 5

In [ ]:
# Display results
from IPython.display import Image, display
import glob
grids = glob.glob('./test_results/*.png')
if grids:
    display(Image(filename=grids[0]))